# Trajectory catalogue

A visual tour of every analytical trajectory defined in
`robot/trajectory.py`.  The catalogue covers the shapes that commonly
stress fusion and SLAM front-ends:

| Trajectory | What it exercises |
|---|---|
| `StraightLineTrajectory` | Zero-acceleration baseline; pure bias-drift check |
| `CircleTrajectory` | Constant centripetal acceleration; circular dead-reckoning |
| `SineWaveTrajectory` | Forward + lateral sinusoidal motion (the original demo path) |
| `ZigzagTrajectory` | Triangle-wave lateral via Fourier; sharp-but-smooth turns |
| `FigureEightTrajectory` | Self-intersecting path; loop-closure stress test |
| `StopAndGoTrajectory` | Drive-stop-drive with smooth ramps; catches IMU drift during stops |

Each trajectory exposes the same interface: `state_at(t) → RobotState`,
returning analytical position, velocity, and acceleration.

In [ ]:
import sys
from pathlib import Path

# This notebook lives in `robot/`; force the repo root to `sys.path[0]`
# so `from robot import ...` resolves to the package next to it, not the
# `robot.py` file inside this directory.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR if (NB_DIR / "sensors").is_dir() else NB_DIR.parent
repo_root_str = str(REPO_ROOT)
while repo_root_str in sys.path:
    sys.path.remove(repo_root_str)
sys.path.insert(0, repo_root_str)

import numpy as np
import matplotlib.pyplot as plt

from robot import (
    TRAJECTORY_CATALOG,
    StraightLineTrajectory,
    CircleTrajectory,
    SineWaveTrajectory,
    ZigzagTrajectory,
    FigureEightTrajectory,
    StopAndGoTrajectory,
)

# A fixed colour per trajectory keeps the plots below readable.
TRAJ_COLORS = {
    "straight": "#1f77b4",
    "circle":   "#ff7f0e",
    "sine":     "#2ca02c",
    "zigzag":   "#d62728",
    "figure8":  "#9467bd",
    "stopgo":   "#8c564b",
}

print("Repo root:", REPO_ROOT)
print("Catalogue:", list(TRAJECTORY_CATALOG))

## 1. Ground-truth paths in the (x, y) plane

Sample each catalogued trajectory at 200 Hz over 20 s and plot its
position trace.  The dot marks the starting point and the colour follows
the legend used in every plot below.

In [ ]:
DURATION_S = 20.0
RATE_HZ = 200.0
DT = 1.0 / RATE_HZ
times = np.arange(0.0, DURATION_S + DT, DT)


def sample(traj) -> dict:
    states = [traj.state_at(float(t)) for t in times]
    return {
        "pos": np.array([s.position for s in states]),
        "vel": np.array([s.velocity for s in states]),
        "acc": np.array([s.acceleration for s in states]),
    }


samples = {name: sample(traj) for name, traj in TRAJECTORY_CATALOG.items()}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (name, data) in zip(axes.flat, samples.items()):
    pos = data["pos"]
    ax.plot(pos[:, 0], pos[:, 1], color=TRAJ_COLORS[name], lw=1.6, label=name)
    ax.plot(pos[0, 0], pos[0, 1], "o", color=TRAJ_COLORS[name], ms=6)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_title(name)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")

fig.suptitle("Trajectory catalogue: position traces", y=1.02)
fig.tight_layout()
plt.show()

## 2. Speed and acceleration magnitudes

Speed and acceleration magnitudes vs time, all trajectories overlaid.
This is the signal an IMU sensor is asked to follow - any trajectory
with non-trivial `|a|` will excite the bias estimator.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for name, data in samples.items():
    speed = np.linalg.norm(data["vel"], axis=1)
    accel = np.linalg.norm(data["acc"], axis=1)
    axes[0].plot(times, speed, color=TRAJ_COLORS[name], lw=1.4, label=name)
    axes[1].plot(times, accel, color=TRAJ_COLORS[name], lw=1.4, label=name)

axes[0].set_ylabel("|v| [m/s]")
axes[0].set_title("Speed magnitude")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="upper right", ncols=3, fontsize=9)

axes[1].set_xlabel("t [s]")
axes[1].set_ylabel("|a| [m/s²]")
axes[1].set_title("Acceleration magnitude")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 3. Parameter sweeps

Each trajectory class is a frozen dataclass so different versions are
trivial to construct.  Two illustrative sweeps:

- **circles** of varying radius (and matched angular speed for constant
  tangential speed),
- **zigzags** with increasing harmonic counts (the path gets sharper as
  more Fourier terms are added, at the cost of bigger acceleration
  spikes at the corners).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# (a) Circle radius sweep, constant tangential speed v = r * omega.
v_tan = 1.0
ax = axes[0]
for r in (0.5, 1.0, 2.0, 4.0):
    traj = CircleTrajectory(radius=r, angular_speed=v_tan / r)
    pos = np.array([traj.state_at(float(t)).position for t in times])
    ax.plot(pos[:, 0], pos[:, 1], lw=1.4, label=f"r={r} m")
ax.set_aspect("equal")
ax.set_title("Circles at constant tangential speed (1 m/s)")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.grid(True, alpha=0.3)
ax.legend()

# (b) Zigzag harmonics sweep, fixed amplitude/period.
ax = axes[1]
short_times = np.linspace(0.0, 8.0, 1601)
for h in (1, 3, 7, 15):
    traj = ZigzagTrajectory(forward_speed=1.0, amplitude=1.0,
                            period_s=4.0, harmonics=h)
    py = np.array([traj.state_at(float(t)).position[1] for t in short_times])
    ax.plot(short_times, py, lw=1.4, label=f"harmonics={h}")
ax.set_title("ZigzagTrajectory: lateral position vs time")
ax.set_xlabel("t [s]")
ax.set_ylabel("y [m]")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

## 4. Numerical sanity check

Cheap correctness test: verify that the analytical accelerations match
finite-difference derivatives of the analytical velocities for every
trajectory.  Errors should be tiny (~1e-9 or smaller).

In [ ]:
def num_deriv(f, t, h=1e-5):
    return (f(t + h) - f(t - h)) / (2.0 * h)


t_probe = 1.7
print(f"{'trajectory':>10}  |v|_anly   |a|_anly   ||a_anly-a_num||")
for name, traj in TRAJECTORY_CATALOG.items():
    s = traj.state_at(t_probe)
    vel_fn = lambda tt: traj.state_at(tt).velocity
    a_num = num_deriv(vel_fn, t_probe)
    a_err = float(np.linalg.norm(a_num - s.acceleration))
    print(f"{name:>10}  {np.linalg.norm(s.velocity):8.4f}   "
          f"{np.linalg.norm(s.acceleration):8.4f}   {a_err:.2e}")